# PersonaPlex + IMTalker Live Avatar Server — RunPod RTX 5090 Deployment

This notebook deploys and launches the **live PersonaPlex + IMTalker AH AudioPace
server** described in `live.md` (cross-checked against `live_working.md` and
`winner_latest_1.md`), the authoritative deployment documents for this stack:

```
PersonaPlex assistant audio + hidden states (12.5 Hz)
-> assistant-output RMS gate (silence Helium seed while not replying)
-> 0.96s lookahead adapter (unitalk_last_layer, 12 hidden steps + 6 future steps)
-> IMTalker generator with static-head LoRA (2h "live_winner" checkpoint)
-> IMTalker FP32 renderer
-> separate Opus audio (/ws/conversation) + JPEG video (/ws/video) websockets
```

`run_live.sh` is a thin wrapper that always launches the **AH AudioPace**
winner (`IMTalker/start_typeah_audio_pace_varm3.sh` ->
`IMTalker/start_winner_live.sh` -> `liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py`),
which fixed an intermittent assistant-audio speed-up after renderer stalls
relative to the earlier AJ NetworkIso server.

It performs, in order:

1. Validates / clones the `speech2avatar` project structure (`IMTalker/`, `personaplex/`,
   `scripts/download_live_assets.sh`, `run_live.sh`, `live.md`, and the AH-winner
   files that must be present in the clone: `start_winner_live.sh`,
   `start_typeah_audio_pace_varm3.sh`, the `AHAudioPace.py` server, the
   `RB_Robert_System_Prompt_full.txt` prompt, and the `_aj_nodrop` HTML page)
   and re-asserts `+x` on every shell script in the checkout.
2. Installs system packages, creates the `python3.11` virtualenv, installs the
   **pinned** `torch==2.8.0+cu128` stack, and all IMTalker / PersonaPlex Python
   dependencies — without ever downgrading PyTorch.
3. Authenticates to Hugging Face from a secure token prompt (never hardcoded).
   Note: `nvidia/personaplex-7b-v1` is a **gated** model repo — accept its terms
   on Hugging Face with the same account before running this notebook.
4. Downloads and **checksum-verifies** every runtime asset: the lookahead/RMS
   adapter, the silence-Helium seed, the current 2-hour static-head LoRA, the
   cached blink motion (currently unused by the AH default launch but still
   downloaded), the base IMTalker checkpoints, the PersonaPlex bnb4 weights,
   the PersonaPlex Mimi/tokenizer files, and the `VARM3` voice prompt.
5. Installs the PersonaPlex/Moshi package with `--no-deps`.
6. Runs full GPU/CUDA/checkpoint/port validation.
7. Launches the live server by invoking the **actual `run_live.sh`** (never
   reimplementing its flags), waits for the documented healthy log markers, and
   confirms the service is reachable on `0.0.0.0:{PORT}` (default `8998`) —
   **`PORT` must match an HTTP port your RunPod pod template actually exposes**,
   or the public proxy URL will never reach it even though the server is healthy.
8. Provides operational cells: log tailing, full diagnostics, and a safe stop switch.

> Run this top-to-bottom on a fresh RunPod RTX 5090 pod (Ubuntu, CUDA 12.8 base image).
> Edit the **Parameters** cell below before running — in particular `PROJECT_ROOT`,
> `PORT` (must match your pod's exposed HTTP port), and, on a brand-new pod with
> no existing clone, `GIT_REPO_URL`.


In [ ]:
!git clone https://github.com/MoshiHead/n_IMTalker_s_Personaplex_v2_remove_unwanted_sound.git speech2avatar

## Parameters

Edit these before running. Leaving an override blank means "use the default
baked into `run_live.sh`" — this notebook never edits that script, it only sets
environment variables that the script already reads.


In [ ]:
import os

# --- Project location -------------------------------------------------------
# If PROJECT_ROOT already contains a valid speech2avatar checkout, it is used
# as-is. Otherwise, if GIT_REPO_URL is set, the repo is cloned into PROJECT_ROOT.
PROJECT_ROOT = "/workspace/speech2avatar"
GIT_REPO_URL = "https://github.com/nash-raf/speech2avatar.git"  # leave blank if already cloned (e.g. by the cell above)
GIT_BRANCH = "main"

# --- Toolchain ---------------------------------------------------------------
VENV_DIR = "/workspace/preprocess_5090"

# --- Service ------------------------------------------------------------------
# PORT must exactly match an "HTTP Port" / "TCP Port" exposed in your RunPod pod
# template. RunPod's proxy (https://<pod-id>-<port>.proxy.runpod.net/) only
# forwards traffic for ports that were actually exposed when the pod was
# created/edited — if the server binds a port your pod doesn't expose, the
# proxy URL will hang or refuse the connection even though the server is
# healthy. Check your pod's "Connect" -> "HTTP Service" list on the RunPod
# dashboard and set PORT to one of those values (commonly 8998 or 8999 for
# this stack; AH's own standalone default is 8998, run_live.sh's own default
# is 8999 if PORT is left unset).
HOST = "0.0.0.0"  # informational only: start_winner_live.sh always binds 0.0.0.0
PORT = 8998

# RunPod sets RUNPOD_POD_ID automatically inside every pod; used only to print
# the exact public proxy URL below. Override manually if it's not auto-detected
# (e.g. running this notebook outside of a RunPod pod).
RUNPOD_POD_ID = os.environ.get("RUNPOD_POD_ID", "")
PUBLIC_URL = f"https://{RUNPOD_POD_ID}-{PORT}.proxy.runpod.net/" if RUNPOD_POD_ID else None

# --- Hugging Face sources (must match live.md / live_working.md / winner_latest_1.md) ---
# Live lookahead/RMS adapter + silence-Helium seed (both in the same dataset repo).
HF_ADAPTER_REPO = "niloy629/hdtf_preprocess"
HF_ADAPTER_FILE = "personaplex_lookahead_rms_adapter/checkpoints/personaplex_lookahead096_future048_rms50_adapter.pt"
HF_SILENCE_FILE = "personaplex_lookahead_rms_adapter/stats/silence_helium_mean.pt"

# Current AH static-head with-audio LoRA (2-hour "live_winner" checkpoint).
HF_LORA_REPO = "niloy629/hdtf_preprocess"
HF_LORA_FILE = "live_winner/lora/ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
HF_LORA_SHA256 = "f34de1bd748857bc802102da578046bb10bd5f664460607d9c785ff35922e52f"

# Cached blink motion-map (downloaded by download_live_assets.sh; not wired into
# the current AH/AJ default launch, kept here so the asset is still verified).
HF_BLINK_REPO = "niloy629/hdtf_preprocess"
HF_BLINK_FILE = "lora/3robert_audio3_ditto_static_motion.pt"
HF_BLINK_SHA256 = "e29a41ff004b228d7efee15cad0f32f4d4bc5466563709e2ba78b158d4e340bb"

HF_IMTALKER_REPO = "cbsjtu01/IMTalker"
HF_IMTALKER_FILES = [
    "config.yaml",
    "renderer.ckpt",
    "generator.ckpt",
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/feature_extractor_config.json",
]

HF_PERSONAPLEX_REPO = "brianmatzelle/personaplex-7b-v1-bnb-4bit"

# Gated repo: accept the terms for nvidia/personaplex-7b-v1 on Hugging Face with
# the same account as HF_TOKEN before running Step 8, or these downloads 403.
HF_PERSONAPLEX_MIMI_REPO = "nvidia/personaplex-7b-v1"
HF_PERSONAPLEX_MIMI_FILES = [
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "tokenizer_spm_32k_3.model",
]
HF_VOICES_FILE = "voices.tgz"  # extracted to PERSONAPLEX_BNB4_DIR/voices/; VARM3.pt is the live default

EXPECTED_TORCH_VERSION = "2.8.0+cu128"

# --- Optional run_live.sh overrides (leave "" to use the script's own default) ---
REF_PATH = ""            # e.g. f"{PROJECT_ROOT}/IMTalker/assets/source_5.png"
A_CFG_SCALE = ""         # e.g. "1.13" (AH default) or "1.15" (AJ default)
NFE = ""                 # e.g. "5" (script default)
PROMPT_FILE = ""         # e.g. f"{PROJECT_ROOT}/IMTalker/prompts/RB_Robert_System_Prompt_full.txt" (script default)
VOICE_PROMPT = ""        # e.g. "VARM3.pt" (script default; use VARM3 unless explicitly told otherwise)
VOICE_PROMPT_DIR = ""    # e.g. "/workspace/voices"
LORA_GENERATOR_PATH = "" # e.g. path to an alternate static-head LoRA checkpoint
DISABLE_LORA = ""        # e.g. "1" to compare the base generator without LoRA

# --- STT + query routing + web search (optional; leave ENABLE_SEARCH=False
# to reproduce the plain conversational launch with zero new flags) ---------
# Pipeline: STT transcribes what the user said -> a small Qwen router decides
# whether the question needs live information -> if yes, web search + compress
# + inject a <ref> block; if no, the model answers from its own knowledge and
# nothing is injected at all. There is no local document index any more.
# The reference LoRA (unmerged PEFT, QLoRA-style on top of the 4-bit
# PersonaPlex base) teaches the model to consume those <lookup>/<ref> tags.
# See start_winner_live.sh for every ROUTER_* / STT_* / COMPRESSOR_* /
# WEB_SEARCH_* override.
ENABLE_SEARCH = True       # True to turn on STT + routing (+ web search below)
WEB_SEARCH_ENABLED = True  # True to let the router actually reach the web (needs WEB_SEARCH_API_KEY, prompted below)
ROUTER_THRESHOLD = "0.40"  # P(needs live data) at/above which a search fires. <0.5 on purpose: an
                           # unnecessary search costs ~2s; a missed one costs a wrong spoken answer.
ROUTER_RULES = "1"         # 1 = instant regex pre-pass before the model (obvious cases cost 0ms)
# ONE small instruct model does double duty: it routes every transcript
# (search / no search) AND compresses web results into one spoken sentence.
# Sharing it is why routing costs no extra VRAM and no extra load time.
COMPRESSOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
COMPRESSOR_DEVICE = "cuda"

# The bundled STT model (kyutai/stt-1b-en_fr) can only produce English and
# French. A transcript in another script is therefore decode garbage, not a
# language surprise -- routing on it answers a question nobody asked and can
# send nonsense to a paid search API. "1" drops such transcripts before the
# router. Set "0" only if you deliberately swap in a multilingual STT model.
STT_REJECT_FOREIGN_SCRIPT = "1"
STT_MAX_NON_LATIN_RATIO = "0.15"   # share of non-Latin LETTERS tolerated before dropping
# The script check above only catches other WRITING SYSTEMS. The STT model is
# bilingual (en/fr) and on unclear audio hallucinates fluent Spanish/French in
# ordinary Latin letters. "1" also drops those (lexical check, microseconds,
# conservative -- it needs positive evidence of another language).
STT_REQUIRE_ENGLISH = "1"
# Caps how far behind real time replies may drift. The GPU producer is pinned to
# real time by frame_q backpressure and can NEVER drain a backlog, so without a
# cap any stall becomes a permanent session-long delay. Oldest audio is dropped.
MAX_INPUT_BUFFER_SEC = "2.0"
# Hold the model silent for the whole search rather than only muting its audio:
# muting alone let it compose an invented number behind the filler and finish
# that sentence even after the real <ref> landed.
SUPPRESS_TEXT_DURING_SEARCH = "1"
# When to mute the model while the route for an utterance is still being decided.
# PersonaPlex is FULL-DUPLEX: it begins its reply within ~1 chunk (80ms) of the
# user going quiet, but routing cannot start until the STT VAD has confirmed
# ~960ms of silence. For that whole second the model used to speak unmuted, so
# the user heard the opening of an ungrounded answer BEFORE the thinking sound
# started -- "I don't have the..." ahead of a gold-price search, "Tesla's..."
# ahead of a stock-price search. Those fragments were also missing from every
# conversation log, because they landed between two turns' slice bounds.
#   "stop"  (default) mute from the first silent frame -> window shrinks to ~80ms
#   "start" mute from the user's first transcribed word (only if a syllable
#           still escapes; longer forced silence biases the model towards quiet)
#   "off"   restore the old behavior
SEARCH_PREHOLD_MODE = "stop"
SEARCH_PREHOLD_MAX_SEC = "8.0"   # safety cap; a decision that never arrives can never mute the avatar for good
# Run the instant regex router on the PARTIAL transcript this many silent frames
# (80ms each) in, so an obvious lookup question starts the thinking sound ~240ms
# after the user stops instead of ~1s. Only the regex tier runs here (no forward
# pass), and a cue raised on a pause that turns out to be mid-sentence is
# cancelled as soon as the user speaks again. 0 disables it.
SEARCH_EARLY_CUE_FRAMES = "3"
# Silence appended after the system prompt. The prompt is force-fed as the
# assistant's OWN speech, so without this the model carries on describing
# itself instead of answering the first question.
PROMPT_SETTLE_SEC = "0.0"   # 0 keeps the opening greeting; see start_winner_live.sh

# Structured conversation logging (user transcripts, router decisions, web
# search, compressor prompts/responses, assistant responses, timing, errors,
# periodic system status). Always active (console prints happen regardless);
# setting a directory here also writes a per-session .log + .jsonl pair there.
CONVERSATION_LOG_DIR = ""   # e.g. f"{PROJECT_ROOT}/conversation_logs" (also the start_winner_live.sh default when ENABLE_SEARCH=1)

# --- Startup wait behaviour ---------------------------------------------------
STARTUP_TIMEOUT_SEC = 900   # PersonaPlex (7B, 4-bit) + IMTalker model load can take a while
POLL_INTERVAL_SEC = 5

# --- Derived paths (must match download_live_assets.sh / run_live.sh) -------
IMTALKER_DIR = f"{PROJECT_ROOT}/IMTalker"
PERSONAPLEX_DIR = f"{PROJECT_ROOT}/personaplex"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"
PERSONAPLEX_BNB4_DIR = f"{CHECKPOINT_DIR}/personaplex_bnb4"
IMTALKER_CKPT_DIR = f"{IMTALKER_DIR}/checkpoints"
ADAPTER_PATH = f"{CHECKPOINT_DIR}/{HF_ADAPTER_FILE}"
SILENCE_HELIUM_PATH = f"{CHECKPOINT_DIR}/{HF_SILENCE_FILE}"
LORA_PATH = f"{CHECKPOINT_DIR}/{HF_LORA_FILE}"
BLINK_MOTION_PATH = f"{CHECKPOINT_DIR}/{HF_BLINK_FILE}"
PERSONAPLEX_WEIGHT_PATH = f"{PERSONAPLEX_BNB4_DIR}/model_bnb_4bit.pt"
VOICE_PROMPT_PATH = f"{PERSONAPLEX_BNB4_DIR}/voices/{VOICE_PROMPT or 'VARM3.pt'}"

# Search-specific derived paths (only used when ENABLE_SEARCH=True).
# The on-disk directory name stays "rag_lora" because that is where
# download_live_assets.sh has always placed this adapter; only its role is
# renamed (it is the <lookup>/<ref> adapter, not a retrieval index).
REF_LORA_DIR = f"{CHECKPOINT_DIR}/rag_lora"
STT_PKG_DIR = f"{CHECKPOINT_DIR}/stt"
if ENABLE_SEARCH and not CONVERSATION_LOG_DIR:
    CONVERSATION_LOG_DIR = f"{PROJECT_ROOT}/conversation_logs"

VENV_PYTHON = f"{VENV_DIR}/bin/python"
VENV_PIP = f"{VENV_DIR}/bin/pip"
VENV_ACTIVATE = f"source {VENV_DIR}/bin/activate"

LOG_PATH = f"{PROJECT_ROOT}/live_server.log"
PID_PATH = f"{PROJECT_ROOT}/.run_live.pid"

os.makedirs("/workspace", exist_ok=True)
print("Parameters loaded.")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"VENV_DIR     = {VENV_DIR}")
print(f"HOST:PORT    = {HOST}:{PORT}")
print(f"ENABLE_SEARCH = {ENABLE_SEARCH}  (web_search={WEB_SEARCH_ENABLED}, router_threshold={ROUTER_THRESHOLD}, rules={ROUTER_RULES})")
if ENABLE_SEARCH:
    print(f"  turn-taking: prehold_mode={SEARCH_PREHOLD_MODE} (max {SEARCH_PREHOLD_MAX_SEC}s), early_cue_frames={SEARCH_EARLY_CUE_FRAMES}")
if ENABLE_SEARCH:
    print(f"CONVERSATION_LOG_DIR = {CONVERSATION_LOG_DIR}")
if RUNPOD_POD_ID:
    print(f"RUNPOD_POD_ID = {RUNPOD_POD_ID}")
    print(f"PUBLIC_URL    = {PUBLIC_URL}")
    print(f"  -> Double-check this port is listed under this pod's Connect -> HTTP Service on the RunPod dashboard.")
else:
    print("[warn] RUNPOD_POD_ID not auto-detected (not running inside a RunPod pod, or the env var is unset).")
    print("       Set RUNPOD_POD_ID manually above if you want the public proxy URL printed automatically.")


## Utilities

Shared helpers used throughout the notebook: a streaming shell runner with
retries, a torch/CUDA probe that always queries the **venv** interpreter (the
notebook kernel itself need not have torch installed), a sha256 checksum
helper, a targeted Hugging Face downloader, and port helpers. These back all
of the "Error Recovery" behaviour required by this deployment.


In [ ]:
import hashlib
import json as _json
import socket
import subprocess
import time


def run(cmd, cwd=None, env=None, check=True, retries=1, retry_delay=8, quiet=False):
    last_returncode = None
    for attempt in range(1, retries + 1):
        if not quiet:
            print(f"$ {cmd}" + (f"   [attempt {attempt}/{retries}]" if retries > 1 else ""))
        proc = subprocess.Popen(
            cmd, shell=True, executable="/bin/bash", cwd=cwd,
            env=env or os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
        last_returncode = proc.returncode
        if last_returncode == 0:
            return 0
        print(f"[warn] command failed with exit code {last_returncode}")
        if attempt < retries:
            print(f"[recovery] retrying in {retry_delay}s...")
            time.sleep(retry_delay)
    if check:
        raise RuntimeError(f"Command failed after {retries} attempt(s) (exit {last_returncode}): {cmd}")
    return last_returncode


def get_torch_info():
    probe = (
        "import torch, json;"
        "print(json.dumps({"
        "'version': torch.__version__,"
        "'cuda_available': torch.cuda.is_available(),"
        "'cuda_version': torch.version.cuda,"
        "'device_name': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
        "}))"
    )
    out = subprocess.run([VENV_PYTHON, "-c", probe], capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout)
        print(out.stderr)
        raise RuntimeError("Failed to query torch/CUDA state inside the venv")
    return _json.loads(out.stdout.strip().splitlines()[-1])


def assert_torch_pin(auto_fix=True):
    info = get_torch_info()
    print(f"[info] torch={info['version']} cuda_available={info['cuda_available']} "
          f"cuda_version={info['cuda_version']} device={info['device_name']}")
    if info["version"] != EXPECTED_TORCH_VERSION:
        print(f"[warn] torch version drifted to '{info['version']}' (expected '{EXPECTED_TORCH_VERSION}')")
        if not auto_fix:
            raise RuntimeError(f"torch pin violated: {info['version']} != {EXPECTED_TORCH_VERSION}")
        print("[recovery] reinstalling pinned torch/torchvision/torchaudio (cu128, no-deps)...")
        run(
            f"{VENV_ACTIVATE} && pip install --force-reinstall --no-deps "
            f"torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
            f"--index-url https://download.pytorch.org/whl/cu128",
            retries=3,
        )
        info = get_torch_info()
        if info["version"] != EXPECTED_TORCH_VERSION:
            raise RuntimeError(f"torch pin recovery failed; still got {info['version']}")
    print(f"[ok] torch pin confirmed: {info['version']}")
    return info


def sha256_of(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def hf_download(repo, filename, local_dir, repo_type="model", retries=3):
    repo_type_flag = f"--repo-type {repo_type}" if repo_type != "model" else ""
    os.makedirs(local_dir, exist_ok=True)
    cmd = (
        f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
        f"hf download {repo} {filename} {repo_type_flag} --local-dir {local_dir}"
    )
    run(cmd, retries=retries, retry_delay=15)


def verify_checksum_with_recovery(path, expected_sha256, repo, filename, repo_type, local_dir, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        if not os.path.exists(path):
            print(f"[fix] missing file, downloading: {path} (attempt {attempt}/{max_attempts})")
            hf_download(repo, filename, local_dir, repo_type)
        actual = sha256_of(path)
        if actual.lower() == expected_sha256.lower():
            print(f"[ok] checksum verified: {path}")
            return True
        print(f"[warn] checksum mismatch for {path}")
        print(f"       expected: {expected_sha256}")
        print(f"       actual:   {actual}")
        print(f"[recovery] deleting and re-downloading (attempt {attempt}/{max_attempts})")
        try:
            os.remove(path)
        except FileNotFoundError:
            pass
        hf_download(repo, filename, local_dir, repo_type)
    raise RuntimeError(f"Checksum verification failed after {max_attempts} attempts: {path}")


def port_listening(host, port, timeout=1.0):
    probe_host = "127.0.0.1" if host == "0.0.0.0" else host
    try:
        with socket.create_connection((probe_host, port), timeout=timeout):
            return True
    except OSError:
        return False


def find_pids_on_port(port):
    out = subprocess.run(
        f"fuser {port}/tcp 2>/dev/null || lsof -t -i:{port} 2>/dev/null",
        shell=True, executable="/bin/bash", capture_output=True, text=True,
    )
    return [p for p in out.stdout.split() if p.strip().isdigit()]


def tail_log(n=200):
    if not os.path.exists(LOG_PATH):
        print(f"[info] no log file yet at {LOG_PATH}")
        return
    with open(LOG_PATH, "r", errors="ignore") as f:
        lines = f.readlines()
    print("".join(lines[-n:]))


print("Utilities loaded.")


## Step 0 — Validate / clone the project structure, fix script permissions

`live.md` / `winner_latest_1.md` expect the layout:

```
speech2avatar/
  IMTalker/
    start_winner_live.sh
    start_typeah_audio_pace_varm3.sh
    liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py
    prompts/RB_Robert_System_Prompt_full.txt
    static/index_v3_binary_fullscreen_aj_nodrop.html
  personaplex/
  scripts/download_live_assets.sh
  run_live.sh
  live.md
```

`winner_latest_1.md` explicitly warns that the AH-winner files were, at one
point, only present in a local working tree and might be missing from a
plain clone of the GitHub remote — so this check verifies them by name
instead of assuming a successful `git clone` is sufficient. If `PROJECT_ROOT`
does not already contain this layout and `GIT_REPO_URL` is set, the repo is
cloned. Otherwise the cell fails fast with a clear diagnostic rather than
proceeding against a broken layout.

This step also runs `chmod +x` on every `*.sh` file in the checkout. In
practice, a clone of this repo can arrive with `IMTalker/start_winner_live.sh`
missing its executable bit; since `start_typeah_audio_pace_varm3.sh` invokes
it with `exec "$SCRIPT_DIR/start_winner_live.sh"` (not `bash ...`), a missing
`+x` bit makes Step 14 fail immediately with `Permission denied` and the live
server process exiting with code 126 — before any Python/CUDA/model code
ever runs.


In [ ]:
REQUIRED_STRUCTURE = [
    "IMTalker",
    "personaplex",
    "scripts/download_live_assets.sh",
    "run_live.sh",
    "live.md",
    "IMTalker/start_winner_live.sh",
    "IMTalker/start_typeah_audio_pace_varm3.sh",
    "IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py",
    "IMTalker/prompts/RB_Robert_System_Prompt_full.txt",
    "IMTalker/static/index_v3_binary_fullscreen_aj_nodrop.html",
]


def structure_ok(root):
    return all(os.path.exists(os.path.join(root, rel)) for rel in REQUIRED_STRUCTURE)


if not structure_ok(PROJECT_ROOT):
    if GIT_REPO_URL:
        print(f"[fix] {PROJECT_ROOT} missing or incomplete; cloning {GIT_REPO_URL} (branch {GIT_BRANCH})")
        os.makedirs(os.path.dirname(PROJECT_ROOT.rstrip("/")) or "/", exist_ok=True)
        if os.path.exists(PROJECT_ROOT) and not os.listdir(PROJECT_ROOT):
            os.rmdir(PROJECT_ROOT)
        run(f"git clone --branch {GIT_BRANCH} {GIT_REPO_URL} {PROJECT_ROOT}", retries=3, retry_delay=10)
    else:
        raise RuntimeError(
            f"PROJECT_ROOT '{PROJECT_ROOT}' does not contain a valid speech2avatar checkout "
            f"(missing one of {REQUIRED_STRUCTURE}) and GIT_REPO_URL is empty. "
            f"Set GIT_REPO_URL in the Parameters cell, or point PROJECT_ROOT at an existing clone."
        )

print(f"Validating structure under {PROJECT_ROOT}:")
status_ok = True
for rel in REQUIRED_STRUCTURE:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    status_ok = status_ok and exists
    print(f"  [{'OK' if exists else 'MISSING'}] {rel}")

if not status_ok:
    raise RuntimeError(
        f"Required speech2avatar structure is incomplete under {PROJECT_ROOT}. "
        f"If IMTalker/start_winner_live.sh or the AHAudioPace.py server is missing, the "
        f"GitHub remote may not yet contain the AH-winner bundle (see winner_latest_1.md) — "
        f"sync those files in manually or update GIT_REPO_URL/GIT_BRANCH."
    )

# Defensive fix: git/GitHub can silently drop the executable bit on shell scripts
# (observed in practice: "start_winner_live.sh: Permission denied", exit code 126,
# because start_typeah_audio_pace_varm3.sh does `exec "$SCRIPT_DIR/start_winner_live.sh"`
# directly rather than `bash start_winner_live.sh`). Re-assert +x on every .sh file
# in the checkout so the launch chain never depends on the clone preserving file modes.
chmod_result = subprocess.run(
    f'find "{PROJECT_ROOT}" -name "*.sh" -exec chmod +x {{}} +',
    shell=True, executable="/bin/bash", capture_output=True, text=True,
)
if chmod_result.returncode != 0:
    print(chmod_result.stdout, chmod_result.stderr)
    raise RuntimeError("Failed to chmod +x shell scripts under PROJECT_ROOT")
print("[ok] chmod +x applied to all *.sh files under PROJECT_ROOT")

print("[ok] project structure validated")


## Step 1 — System dependencies

Matches `live.md`'s *Fresh Pod Setup* exactly.


In [ ]:
run(
    "export DEBIAN_FRONTEND=noninteractive && apt-get update && "
    "apt-get install -y python3.11 python3.11-venv ffmpeg git htop tmux",
    retries=2, retry_delay=10,
)


## Step 2 — GPU / driver check (pre-PyTorch)

Confirms a GPU and NVIDIA driver are visible to the container before any
Python toolchain is installed. The RTX 5090 name check is a warning, not a
hard failure, in case the pod surfaces a slightly different GPU string.


In [ ]:
smi = subprocess.run("nvidia-smi", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(smi.stdout)
if smi.returncode != 0:
    print(smi.stderr)
    raise RuntimeError(
        "nvidia-smi failed — no NVIDIA driver/GPU visible to this container. "
        "Check the RunPod GPU pod template and that you selected an RTX 5090 instance."
    )

if "5090" in smi.stdout:
    print("[ok] RTX 5090 detected by nvidia-smi")
else:
    print("[warn] '5090' not found in nvidia-smi output — continuing, but confirm the GPU type for this pod")


## Step 3 — Python 3.11 virtual environment


In [ ]:
run(f"python3.11 -m venv {VENV_DIR}")
run(f"{VENV_ACTIVATE} && python -m pip install --upgrade pip wheel")
run(f'{VENV_ACTIVATE} && python -m pip install "setuptools==80.9.0"')
print(f"[ok] venv ready at {VENV_DIR}")


## Step 4 — Pinned PyTorch (cu128)

PyTorch must remain exactly `2.8.0+cu128` for the rest of this deployment.
Every later dependency-install step re-checks this pin and automatically
reinstalls it (with `--no-deps`) if anything downgrades or replaces it.


In [ ]:
run(
    f"{VENV_ACTIVATE} && pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
    f"--index-url https://download.pytorch.org/whl/cu128",
    retries=3, retry_delay=20,
)


In [ ]:
torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError(
        "torch.cuda.is_available() is False inside the venv. "
        "Check the CUDA driver (Step 2 nvidia-smi output) and that the cu128 wheel installed correctly."
    )
print(f"[ok] CUDA available, device = {torch_info['device_name']}")


## Step 5 — IMTalker Python dependencies


In [ ]:
run(f"{VENV_ACTIVATE} && pip install -r {IMTALKER_DIR}/requirement.txt", cwd=PROJECT_ROOT, retries=2, retry_delay=15)
assert_torch_pin()

check = subprocess.run(f"{VENV_ACTIVATE} && pip check", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(check.stdout or "[ok] no dependency conflicts reported")
if check.returncode != 0 and check.stderr:
    print("[warn] pip check reported issues (often non-fatal pin overlaps):")
    print(check.stderr)


## Step 6 — Hugging Face tooling and streaming/codec dependencies


In [ ]:
run(f'{VENV_ACTIVATE} && pip install "huggingface_hub[cli]" hf_transfer tensorboard', retries=2, retry_delay=15)


In [ ]:
run(
    f'{VENV_ACTIVATE} && pip install "sphn>=0.2.0,<0.3.0" einops sentencepiece aiohttp av aiortc bitsandbytes',
    retries=2, retry_delay=15,
)
assert_torch_pin()


## Step 6b - STT / routing / web-search dependencies (skipped if `ENABLE_SEARCH=False`)

Installs `transformers==4.52.4` *last*, deliberately overriding the `4.30.2`
pinned by `IMTalker/requirement.txt` in Step 5 - needed for the STT submodel
and for the Qwen model that both routes queries and compresses web results.
IMTalker's own `transformers` usage (`Wav2Vec2FeatureExtractor`) is a
long-stable API; still worth a smoke test after Step 15 confirms the server is
healthy. Also re-pins `aiohttp<3.11` (the web-search code path caps it there)
and installs an *isolated* copy of the genuine upstream Kyutai `moshi` package
for the STT/VAD submodel only - it is never installed into the venv's normal
site-packages, so it can't collide with the PersonaPlex fork's own `moshi`
import (see `IMTalker/search_helpers.py` for why that collision would
otherwise happen).

**No longer installed:** `faiss-cpu` and `sentence-transformers`. Both existed
only for the local-document index, which this pipeline no longer has. The
router uses the Qwen model already loaded for compression, so routing costs no
extra download, no extra VRAM, and no extra load time.


In [ ]:
if ENABLE_SEARCH:
    # faiss-cpu and sentence-transformers are deliberately NOT installed:
    # they existed only for the removed local-document index. peft is needed
    # for the <lookup>/<ref> LoRA, transformers for the STT submodel and the
    # Qwen router/compressor.
    run(
        f'{VENV_ACTIVATE} && pip install "peft>=0.19,<0.20" "transformers==4.52.4"',
        retries=2, retry_delay=15,
    )
    run(f'{VENV_ACTIVATE} && pip install "aiohttp>=3.10.5,<3.11"', retries=2, retry_delay=15)
    assert_torch_pin()

    os.makedirs(STT_PKG_DIR, exist_ok=True)
    run(
        f'{VENV_ACTIVATE} && pip install --no-deps --target {STT_PKG_DIR} moshi',
        retries=2, retry_delay=15,
    )
    print(f"[ok] isolated upstream moshi (STT) installed at {STT_PKG_DIR}")
else:
    print("ENABLE_SEARCH is False - skipping STT/routing dependency install.")


## Step 7 — Hugging Face authentication

Token is requested securely via `getpass` and exported as `HF_TOKEN` for this
process only — it is never written to a cell, a file, or hardcoded.

> **Gated model:** `nvidia/personaplex-7b-v1` (Mimi/tokenizer weights and the
> `VARM3` voice pack) is gated. Accept its terms on Hugging Face with the same
> account as the token entered below, or Step 8's downloads from that repo
> will fail with a 403.


In [ ]:
import getpass

authenticated = False
for attempt in range(1, 4):
    # token = getpass.getpass(f"HF_TOKEN (attempt {attempt}/3, input hidden): ").strip()
    token = '_aKHzCnOPrapIEwfnuxyQUszisfBqaRrTXQ'
    if not token:
        print("[warn] empty token entered, try again")
        continue
    os.environ["HF_TOKEN"] = token
    run(f'{VENV_ACTIVATE} && hf auth login --token "$HF_TOKEN"', check=False)
    whoami = subprocess.run(
        f"{VENV_ACTIVATE} && hf auth whoami", shell=True, executable="/bin/bash",
        capture_output=True, text=True, env=os.environ.copy(),
    )
    if whoami.returncode == 0 and whoami.stdout.strip():
        print(f"[ok] authenticated to Hugging Face as: {whoami.stdout.strip().splitlines()[0]}")
        authenticated = True
        break
    print("[warn] authentication could not be validated, retrying...")
    print(whoami.stdout, whoami.stderr)

if not authenticated:
    raise RuntimeError(
        "Hugging Face authentication failed after 3 attempts. "
        "Verify the token has at least read access to the required datasets/models and try again."
    )


In [ ]:
# Web-search API key (Tavily/Serper/Bing) - requested securely via getpass,
# same pattern as HF_TOKEN above. NEVER hardcode this in a cell: notebooks get
# committed, shared, and pasted into issues, and a key pasted here leaks with
# them. Only asked for if both ENABLE_SEARCH and WEB_SEARCH_ENABLED are True.
if ENABLE_SEARCH and WEB_SEARCH_ENABLED:
    import getpass as _getpass

    web_search_key = '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'
    if not web_search_key:
        web_search_key = _getpass.getpass(
            "Web search API key for the provider set in start_winner_live.sh's "
            "WEB_SEARCH_PROVIDER (default tavily), input hidden: "
        ).strip()
    if not web_search_key:
        raise RuntimeError("WEB_SEARCH_ENABLED is True but no web search API key was provided.")
    os.environ["WEB_SEARCH_API_KEY"] = web_search_key
    print("[ok] web search API key set for this process (never printed, never written to a cell).")
else:
    print("Web search disabled or search disabled - skipping web-search key prompt.")


## Step 8 — Download runtime assets

Runs the project's own `scripts/download_live_assets.sh` (the authoritative
asset list — now a 10-step download covering the base IMTalker checkpoints,
the lookahead/RMS adapter, the silence-Helium seed, the current 2-hour
static-head LoRA, the cached blink motion, the PersonaPlex bnb4 weights, the
gated PersonaPlex Mimi/tokenizer files, and the `VARM3` voice pack), with
automatic retry on interrupted/failed downloads.


In [ ]:
run(
    f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
    f"export SPEECH2AVATAR_ROOT={PROJECT_ROOT} && "
    f"bash scripts/download_live_assets.sh",
    cwd=PROJECT_ROOT, retries=3, retry_delay=20,
)


## Step 9 — Checksum verification (LoRA + cached blink motion)

Both files are re-downloaded automatically on checksum mismatch.


In [ ]:
verify_checksum_with_recovery(
    LORA_PATH, HF_LORA_SHA256, HF_LORA_REPO, HF_LORA_FILE, "dataset", CHECKPOINT_DIR,
)
verify_checksum_with_recovery(
    BLINK_MOTION_PATH, HF_BLINK_SHA256, HF_BLINK_REPO, HF_BLINK_FILE, "dataset", CHECKPOINT_DIR,
)


## Step 10 — Verify remaining required assets

Base IMTalker checkpoints, the wav2vec frontend, the lookahead/RMS adapter,
the silence-Helium seed, the PersonaPlex bnb4 weights, the PersonaPlex
Mimi/tokenizer files, and the `VARM3` voice prompt (shipped inside
`voices.tgz`, so it is downloaded and extracted separately from the other
single-file assets). Anything missing is re-downloaded by name from the
correct repo before the table is reprinted.


In [ ]:
import tarfile

REQUIRED_ASSETS = []
for fname in HF_IMTALKER_FILES:
    REQUIRED_ASSETS.append({
        "path": os.path.join(IMTALKER_CKPT_DIR, fname),
        "repo": HF_IMTALKER_REPO, "filename": fname, "repo_type": "model", "local_dir": IMTALKER_CKPT_DIR,
    })
REQUIRED_ASSETS.append({
    "path": ADAPTER_PATH, "repo": HF_ADAPTER_REPO, "filename": HF_ADAPTER_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": SILENCE_HELIUM_PATH, "repo": HF_ADAPTER_REPO, "filename": HF_SILENCE_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": LORA_PATH, "repo": HF_LORA_REPO, "filename": HF_LORA_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": BLINK_MOTION_PATH, "repo": HF_BLINK_REPO, "filename": HF_BLINK_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": PERSONAPLEX_WEIGHT_PATH, "repo": HF_PERSONAPLEX_REPO, "filename": "model_bnb_4bit.pt",
    "repo_type": "model", "local_dir": PERSONAPLEX_BNB4_DIR,
})
for fname in HF_PERSONAPLEX_MIMI_FILES:
    REQUIRED_ASSETS.append({
        "path": os.path.join(PERSONAPLEX_BNB4_DIR, fname),
        "repo": HF_PERSONAPLEX_MIMI_REPO, "filename": fname, "repo_type": "model", "local_dir": PERSONAPLEX_BNB4_DIR,
    })

for asset in REQUIRED_ASSETS:
    if not os.path.exists(asset["path"]):
        print(f"[fix] missing asset, downloading: {asset['path']}")
        hf_download(asset["repo"], asset["filename"], asset["local_dir"], asset["repo_type"])

# The VARM3 voice prompt ships inside voices.tgz (nvidia/personaplex-7b-v1), not
# as a standalone HF file, so it needs a download + extract step of its own.
if not os.path.exists(VOICE_PROMPT_PATH):
    print(f"[fix] missing voice prompt, downloading + extracting: {HF_VOICES_FILE}")
    hf_download(HF_PERSONAPLEX_MIMI_REPO, HF_VOICES_FILE, PERSONAPLEX_BNB4_DIR, "model")
    voices_tgz = os.path.join(PERSONAPLEX_BNB4_DIR, HF_VOICES_FILE)
    with tarfile.open(voices_tgz) as tf:
        tf.extractall(PERSONAPLEX_BNB4_DIR)

print("Required asset status:")
all_present = True
for asset in REQUIRED_ASSETS:
    exists = os.path.exists(asset["path"])
    all_present = all_present and exists
    size = f"{os.path.getsize(asset['path']) / (1 << 20):.1f} MB" if exists else "-"
    print(f"  [{'OK' if exists else 'MISSING'}] {asset['path']}  {size}")

voice_exists = os.path.exists(VOICE_PROMPT_PATH)
all_present = all_present and voice_exists
voice_size = f"{os.path.getsize(VOICE_PROMPT_PATH) / (1 << 20):.1f} MB" if voice_exists else "-"
print(f"  [{'OK' if voice_exists else 'MISSING'}] {VOICE_PROMPT_PATH}  {voice_size}")

if not all_present:
    raise RuntimeError("One or more required assets are still missing after automatic recovery; see table above")

print("[ok] all required runtime assets present")


## Step 11 — Install PersonaPlex / Moshi (`--no-deps`)

Per `live.md`, the Moshi package source ships inside the downloaded
`personaplex_bnb4` weights folder. As a recovery fallback (in case a given
weights snapshot does not bundle it), this also checks the repo's own
`personaplex/moshi` checkout. Either way the install uses `--no-deps` so the
pinned PyTorch stack from Step 4 is never touched.


In [ ]:
moshi_candidates = [
    os.path.join(PERSONAPLEX_BNB4_DIR, "moshi"),
    os.path.join(PERSONAPLEX_DIR, "moshi"),
]
moshi_dir = next((d for d in moshi_candidates if os.path.isdir(d)), None)
if moshi_dir is None:
    raise RuntimeError(
        f"Could not find a moshi/ package source in any of: {moshi_candidates}. "
        f"Re-run Step 8 download, or confirm the PersonaPlex bnb4 snapshot bundles moshi/."
    )
if moshi_dir != moshi_candidates[0]:
    print(f"[recovery] moshi/ not found under personaplex_bnb4; using fallback source at {moshi_dir}")

run(f"{VENV_ACTIVATE} && pip install -e {moshi_dir} --no-deps", retries=2, retry_delay=10)
assert_torch_pin(auto_fix=True)
print("[ok] PersonaPlex/Moshi installed without modifying the pinned torch stack")


## Step 11b - Verify search assets (skipped if `ENABLE_SEARCH=False`)

`scripts/download_live_assets.sh` (Step 8) already fetched the reference LoRA
(`REF_LORA_DIR`, downloaded from `Darknsu/helium_lora_v1` with a hand-written
`adapter_config.json`, matching the source project's own trick since that
dataset repo doesn't publish one). That adapter teaches the model to consume
the injected `<lookup>`/`<ref>` tags, so it is still required even though the
referenced text now comes from a web search rather than a document index.

This cell confirms it is present before launch, checks that
`IMTalker/search_helpers.py` and `IMTalker/conversation_logger.py` both exist
*and import cleanly inside the venv*, and asserts the resolved `moshi`
package's `get_moshi_lm` still declares `quantize_4bit` (it comes from the
downloaded bnb-4bit snapshot, not from this repo, and should never be silently
overwritten).

There is no `rag_index/` check any more - the local document index was removed.


In [ ]:
if ENABLE_SEARCH:
    ref_lora_config = os.path.join(REF_LORA_DIR, "lora", "adapter_config.json")
    search_helpers_py = os.path.join(IMTALKER_DIR, "search_helpers.py")
    conv_logger_py = os.path.join(IMTALKER_DIR, "conversation_logger.py")
    missing_search = [
        p for p in (ref_lora_config, search_helpers_py, conv_logger_py)
        if not os.path.exists(p)
    ]
    if missing_search:
        raise RuntimeError(
            f"ENABLE_SEARCH is True but required assets/files are missing: {missing_search}. "
            f"If IMTalker/search_helpers.py or IMTalker/conversation_logger.py is missing, the "
            f"checked-out repo does not contain the routing/search code -- this is the most common "
            f"cause of search silently degrading to 'search disabled' deep in the log with no "
            f"obvious error. Confirm those two files were actually committed/pushed to the repo "
            f"this notebook clones. Otherwise re-run Step 8 (scripts/download_live_assets.sh), "
            f"or set ENABLE_SEARCH = False."
        )
    print(f"[ok] reference LoRA present: {ref_lora_config}")
    print(f"[ok] IMTalker/search_helpers.py present")
    print(f"[ok] IMTalker/conversation_logger.py present")

    quantize_4bit_check = subprocess.run(
        [VENV_PYTHON, "-c",
         "import sys, inspect; sys.path.insert(0, sys.argv[1]); from moshi.models import loaders; "
         "print(\'quantize_4bit\' in inspect.signature(loaders.get_moshi_lm).parameters)",
         f"{PERSONAPLEX_BNB4_DIR}/moshi"],
        capture_output=True, text=True,
    )
    print(quantize_4bit_check.stdout.strip(), quantize_4bit_check.stderr.strip())
    if quantize_4bit_check.stdout.strip() != "True":
        raise RuntimeError(
            "The resolved moshi package's get_moshi_lm no longer declares quantize_4bit - "
            "the downloaded personaplex_bnb4 snapshot may have changed shape. "
            "Do not proceed without checking IMTalker/liveTry.py's loader compatibility."
        )
    print("[ok] quantize_4bit still supported by the resolved moshi loader")

    # Importability check: catches search_helpers.py/conversation_logger.py
    # existing on disk but failing to import inside the actual venv (missing
    # dependency, syntax error, etc.) -- exactly the failure class that
    # previously showed up only as a silent "disabled" deep in the server log
    # with no easy way to find the real cause. Runs in the SAME venv the live
    # server itself uses.
    _import_probe = (
        "import sys, traceback\n"
        f"sys.path.insert(0, {IMTALKER_DIR!r})\n"
        "try:\n"
        "    import search_helpers, conversation_logger\n"
        "    print(\'IMPORT_OK\')\n"
        "except Exception:\n"
        "    traceback.print_exc()\n"
        "    print(\'IMPORT_FAILED\')\n"
    )
    import_check = subprocess.run([VENV_PYTHON, "-c", _import_probe], capture_output=True, text=True)
    print(import_check.stdout)
    print(import_check.stderr)
    if "IMPORT_OK" not in import_check.stdout:
        raise RuntimeError(
            "IMTalker/search_helpers.py and/or conversation_logger.py exist but failed to import "
            "in the venv (see traceback above) -- fix this before launching, or search will "
            "silently disable itself at server startup."
        )
    print("[ok] search_helpers / conversation_logger import cleanly in the venv")

    # "Thinking sound" -- audible cue played while a routing decision or web
    # search is in flight (see IMTalker/liveTry...AHAudioPace.py's _start_turn
    # / _step). Not a hard requirement (the avatar still works without it,
    # just silently/near-silently through the wait), but a missing file here
    # is exactly what caused it to be silent with zero explanation last time
    # -- *.wav is gitignored by default in this repo, so this asset needs an
    # explicit negation entry to survive a git push/clone (already added to
    # .gitignore and personaplex/.gitignore).
    thinking_sound_path = os.path.join(PROJECT_ROOT, "personaplex", "ai-thinking-sound.wav")
    if os.path.exists(thinking_sound_path):
        print(f"[ok] thinking-sound WAV present: {thinking_sound_path}")
    else:
        print(
            f"[warn] thinking-sound WAV NOT FOUND at {thinking_sound_path} -- "
            f"the avatar will be silent (or near-silent) while routing/searching "
            f"instead of playing the thinking cue. This is almost always a git problem: "
            f"*.wav is gitignored in this repo by default, so the file must be force-added "
            f"or explicitly un-ignored to survive a push. Check the repo this notebook "
            f"cloned actually contains personaplex/ai-thinking-sound.wav; if not, add it "
            f"with `git add -f personaplex/ai-thinking-sound.wav` (or the .gitignore negation "
            f"`!ai-thinking-sound.wav`) on the machine you push from, then re-clone/re-pull here."
        )

else:
    print("ENABLE_SEARCH is False - skipping search asset verification.")


## Step 11c - Component self-tests (skipped if `ENABLE_SEARCH=False`)

Each cell below loads exactly one component in isolation, in the same venv the
live server uses, and prints either `[ok] ...` lines or a full Python
traceback. Run these BEFORE launching (Step 14) to confirm each piece actually
works, rather than only discovering a silent failure deep in
`live_server.log` after the avatar is already running. These are read-only
sanity checks - they do not affect the live server process.

Order: **router** (the new decision component), **STT**, **compressor**,
**web search**. The router test is the important one to read - it prints the
actual search / no-search verdict for a set of sample questions, so you can
see the behavior you will get before a single word is spoken.


In [ ]:
if ENABLE_SEARCH:
    # Router self-test. Loads the Qwen model once, then prints the verdict for
    # a spread of questions that SHOULD need live data and questions that
    # should NOT. Read the table: it is the clearest preview of how the
    # assistant will behave. Tune ROUTER_THRESHOLD in the params cell if the
    # split is not where you want it.
    _probe = f"""
import sys, traceback
sys.path.insert(0, {IMTALKER_DIR!r})
try:
    import search_helpers

    # 1) Rules-only pass -- no model, microseconds, no GPU needed.
    #    Four ordered layers: explicit time > static question shape >
    #    live topic > chitchat. Shape beats topic, which is what stops
    #    "how do I invest in Bitcoin" being treated as a market lookup.
    rule_cases = [
        ("what is the gold price today", True),
        ("who won the match yesterday", True),
        ("tell me the latest news", True),
        ("how much is an ounce of gold", True),
        ("how do I invest in Bitcoin", False),
        ("how I can invest in cryptocurrency?", False),
        ("is it safe to invest in cryptocurrency?", False),
        ("how does the stock market work", False),
        ("what is the difference between stocks and bonds", False),
        ("should I buy gold", False),
        ("when was Bitcoin created", False),
        ("thank you so much", False),
        ("what is Bitcoin?", None),
    ]
    print("--- Tier 0: layered instant rules ---")
    for q, expected in rule_cases:
        got, why = search_helpers.rule_route_explain(q)
        label = {{True: "SEARCH", False: "no search", None: "-> ask the model"}}[got]
        flag = "ok " if got == expected else "DIFF"
        print(f"  [{{flag}}] {{label:16s}} {{q!r}}")
        print(f"           {{why}}")

    # 2) Full router incl. the Qwen forward pass.
    cc = search_helpers.ContextCompressor(
        model_name={COMPRESSOR_MODEL!r}, device="cuda", quantize_4bit=True,
    )
    router = search_helpers.QueryRouter.from_compressor(
        cc, threshold=float({ROUTER_THRESHOLD!r}), use_rules=bool(int({ROUTER_RULES!r})),
    )
    print("")
    print("--- Tier 1: full router (rules + model) ---")
    cases = [
        "what is the gold price today",
        "what is the weather in Dhaka right now",
        "who is the current president of France",
        "what happened in the news this week",
        "explain how a transformer neural network works",
        "what is the capital of Japan",
        "tell me a joke",
        "how do I write a for loop in python",
        "is Tesla a good investment",
        "what is the population of Tokyo",
        # the shapes that previously misrouted -- these should now be
        # settled by the rules, never reaching the model at all
        "how I can invest in cryptocurrency?",
        "is it safe to invest in cryptocurrency?",
        "what is Bitcoin?",
    ]
    print(f"  {{'verdict':<12}} {{'via':<7}} {{'score':>6}}  {{'ms':>6}}  question")
    for q in cases:
        v = router.decide(q)
        verdict = "SEARCH" if v["needs_search"] else "no search"
        print(f"  {{verdict:<12}} {{v['source']:<7}} {{v['score']:>6.3f}}  "
              f"{{1000*v['elapsed_s']:>6.0f}}  {{q}}")
    print("")
    print("ROUTER_SELFTEST_PASSED")
except Exception:
    traceback.print_exc()
    print("ROUTER_SELFTEST_FAILED")
"""
    result = subprocess.run([VENV_PYTHON, "-c", _probe], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if "ROUTER_SELFTEST_PASSED" not in result.stdout:
        print("[FAIL] Router self-test failed -- see traceback above. Without the router, "
              "every turn is answered from the model's own knowledge and nothing is ever searched.")
else:
    print("ENABLE_SEARCH is False - skipping.")


In [ ]:
if ENABLE_SEARCH:
    _probe = f"""
import sys, traceback
sys.path.insert(0, {IMTALKER_DIR!r})
try:
    import search_helpers, torch
    moshi_stt = search_helpers.load_upstream_moshi_stt({STT_PKG_DIR!r})
    print("[ok] upstream moshi package loaded under alias moshi_stt (version:", getattr(moshi_stt, "__version__", "?"), ")")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    stt_info = moshi_stt.models.loaders.CheckpointInfo.from_hf_repo("kyutai/stt-1b-en_fr-candle")
    print("[ok] CheckpointInfo.from_hf_repo resolved")
    stt_mimi = stt_info.get_mimi(device=device)
    stt_lm = stt_info.get_moshi(device=device, dtype=torch.bfloat16)
    n_params = sum(p.numel() for p in stt_lm.parameters()) / 1e9
    print(f"[ok] STT model loaded: {{n_params:.2f}}B params on {{device}}")
    lm_gen = moshi_stt.models.LMGen(stt_lm, temp=0, temp_text=0.0)
    print("[ok] STT LMGen constructed:", hasattr(lm_gen, "step_with_extra_heads"))
    print("STT_SELFTEST_PASSED")
except Exception:
    traceback.print_exc()
    print("STT_SELFTEST_FAILED")
"""
    result = subprocess.run([VENV_PYTHON, "-c", _probe], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if "STT_SELFTEST_PASSED" not in result.stdout:
        print("[FAIL] STT self-test failed -- see traceback above. Turn detection and routing will not work without this.")
else:
    print("ENABLE_SEARCH is False - skipping.")


In [ ]:
if ENABLE_SEARCH:
    _probe = """
import sys, traceback
sys.path.insert(0, r'{imtalker}')
try:
    import search_helpers
    cc = search_helpers.ContextCompressor(model_name="{model}", device="{device}", quantize_4bit=True)
    result = cc.compress(
        question="What is the deferred tax liability?",
        chunks=[{{"id": "warmup", "source": "warmup", "text": "Deferred tax liability is 1.01.", "similarity_score": 1.0}}],
    )
    print(f"[ok] Qwen compressor loaded and responded: {{result!r}}")
    print("COMPRESSOR_SELFTEST_PASSED")
except Exception:
    traceback.print_exc()
    print("COMPRESSOR_SELFTEST_FAILED")
""".format(imtalker=IMTALKER_DIR, model=COMPRESSOR_MODEL, device=COMPRESSOR_DEVICE)
    result = subprocess.run([VENV_PYTHON, "-c", _probe], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if "COMPRESSOR_SELFTEST_PASSED" not in result.stdout:
        print("[FAIL] Compressor self-test failed -- see traceback above. The router shares this model, so BOTH routing and search are disabled if it cannot load: every turn would then be answered from the model's own knowledge.")
else:
    print("ENABLE_SEARCH is False - skipping.")


In [ ]:
if ENABLE_SEARCH and WEB_SEARCH_ENABLED:
    _probe = f"""
import sys, traceback, os
sys.path.insert(0, {IMTALKER_DIR!r})
try:
    import search_helpers
    key = os.environ.get("WEB_SEARCH_API_KEY", "")
    if not key:
        print("[skip] WEB_SEARCH_API_KEY not set in this process's environment")
    else:
        hits = search_helpers.web_search_query_sync("current weather in New York", key, "tavily", 3, 5.0)
        print(f"[ok] web search returned {{len(hits)}} hits")
        for h in hits:
            print(f"    score={{h['similarity_score']:.3f}} source={{h['source']}} text={{h['text'][:80]!r}}")
    print("WEB_SEARCH_SELFTEST_PASSED")
except Exception:
    traceback.print_exc()
    print("WEB_SEARCH_SELFTEST_FAILED")
"""
    result = subprocess.run([VENV_PYTHON, "-c", _probe], capture_output=True, text=True, env=os.environ.copy())
    print(result.stdout)
    print(result.stderr)
    if "WEB_SEARCH_SELFTEST_PASSED" not in result.stdout:
        print("[FAIL] Web search self-test failed -- see traceback/output above.")
else:
    print("ENABLE_SEARCH or WEB_SEARCH_ENABLED is False - skipping.")


## Step 12 — Pre-launch validation

GPU, CUDA, the full asset table, and port availability (with automatic
recovery if a *previous run of this same notebook* is still holding the port).


In [ ]:
def check_port_and_recover(port):
    if not port_listening("127.0.0.1", port):
        print(f"[ok] port {port} is free")
        return
    pids_on_port = find_pids_on_port(port)
    prior_pid = None
    if os.path.exists(PID_PATH):
        prior_pid = open(PID_PATH).read().strip()
    if prior_pid and prior_pid in pids_on_port:
        print(f"[recovery] port {port} held by a previous notebook-launched server (pid {prior_pid}); terminating it")
        subprocess.run(["kill", "-9", prior_pid])
        time.sleep(2)
        if port_listening("127.0.0.1", port):
            raise RuntimeError(f"Port {port} still in use after terminating prior pid {prior_pid}")
        print(f"[ok] port {port} freed")
    else:
        raise RuntimeError(
            f"Port {port} is already in use by pid(s) {pids_on_port}, which were not started by this "
            f"notebook. Stop that process manually (or choose a different PORT) before launching."
        )


check_port_and_recover(PORT)

torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError("CUDA not available inside venv torch at launch time")
if torch_info["device_name"] and "5090" in torch_info["device_name"]:
    print(f"[ok] GPU confirmed: {torch_info['device_name']}")
else:
    print(f"[warn] GPU device name '{torch_info['device_name']}' does not mention 5090 — continuing anyway")

missing = [a["path"] for a in REQUIRED_ASSETS if not os.path.exists(a["path"])]
if not os.path.exists(VOICE_PROMPT_PATH):
    missing.append(VOICE_PROMPT_PATH)
if missing:
    raise RuntimeError(f"Missing required assets immediately before launch: {missing}")

print("[ok] pre-launch validation passed")


## Step 13 — Inspect `run_live.sh` and resolve runtime overrides

The notebook never reimplements `run_live.sh`'s argument list — it reads and
prints the actual script, then prepares only the environment-variable
overrides the script (and the `start_winner_live.sh` it delegates to) already
support: `REF_PATH`, `A_CFG_SCALE`, `NFE`, `PROMPT_FILE`, `VOICE_PROMPT`,
`VOICE_PROMPT_DIR`, `LORA_GENERATOR_PATH`, `DISABLE_LORA`, plus
`HOST`/`PORT`/`VENV_DIR`.

> The new `run_live.sh` reads `VENV_DIR` (not the old `VENV`), and it no
> longer accepts a raw `TEXT_PROMPT` string — the system prompt is always
> read from a file (`PROMPT_FILE`, defaulting to
> `IMTalker/prompts/RB_Robert_System_Prompt_full.txt`).


In [ ]:
with open(os.path.join(PROJECT_ROOT, "run_live.sh")) as f:
    run_live_sh_contents = f.read()
print(run_live_sh_contents)


In [ ]:
env_overrides = {
    "HOST": HOST,
    "PORT": str(PORT),
    "VENV_DIR": VENV_DIR,
    "SPEECH2AVATAR_ROOT": PROJECT_ROOT,
}
if REF_PATH:
    env_overrides["REF_PATH"] = REF_PATH
if A_CFG_SCALE:
    env_overrides["A_CFG_SCALE"] = str(A_CFG_SCALE)
if NFE:
    env_overrides["NFE"] = str(NFE)
if PROMPT_FILE:
    env_overrides["PROMPT_FILE"] = PROMPT_FILE
if VOICE_PROMPT:
    env_overrides["VOICE_PROMPT"] = VOICE_PROMPT
if VOICE_PROMPT_DIR:
    env_overrides["VOICE_PROMPT_DIR"] = VOICE_PROMPT_DIR
if LORA_GENERATOR_PATH:
    env_overrides["LORA_GENERATOR_PATH"] = LORA_GENERATOR_PATH
if DISABLE_LORA:
    env_overrides["DISABLE_LORA"] = str(DISABLE_LORA)

if ENABLE_SEARCH:
    env_overrides["ENABLE_SEARCH"] = "1"
    env_overrides["REF_LORA_DIR"] = REF_LORA_DIR
    env_overrides["STT_PKG_DIR"] = STT_PKG_DIR
    env_overrides["CONVERSATION_LOG_DIR"] = CONVERSATION_LOG_DIR
    env_overrides["ROUTER_THRESHOLD"] = str(ROUTER_THRESHOLD)
    env_overrides["ROUTER_RULES"] = str(ROUTER_RULES)
    env_overrides["COMPRESSOR_MODEL"] = COMPRESSOR_MODEL
    env_overrides["COMPRESSOR_DEVICE"] = COMPRESSOR_DEVICE
    env_overrides["STT_REJECT_FOREIGN_SCRIPT"] = str(STT_REJECT_FOREIGN_SCRIPT)
    env_overrides["STT_MAX_NON_LATIN_RATIO"] = str(STT_MAX_NON_LATIN_RATIO)
    env_overrides["STT_REQUIRE_ENGLISH"] = str(STT_REQUIRE_ENGLISH)
    env_overrides["MAX_INPUT_BUFFER_SEC"] = str(MAX_INPUT_BUFFER_SEC)
    env_overrides["SUPPRESS_TEXT_DURING_SEARCH"] = str(SUPPRESS_TEXT_DURING_SEARCH)
    env_overrides["SEARCH_PREHOLD_MODE"] = str(SEARCH_PREHOLD_MODE)
    env_overrides["SEARCH_PREHOLD_MAX_SEC"] = str(SEARCH_PREHOLD_MAX_SEC)
    env_overrides["SEARCH_EARLY_CUE_FRAMES"] = str(SEARCH_EARLY_CUE_FRAMES)
    env_overrides["PROMPT_SETTLE_SEC"] = str(PROMPT_SETTLE_SEC)
    if WEB_SEARCH_ENABLED:
        env_overrides["WEB_SEARCH_ENABLED"] = "1"
        env_overrides["WEB_SEARCH_API_KEY"] = os.environ.get("WEB_SEARCH_API_KEY", "")
    else:
        # Explicit 0 so start_winner_live.sh does not auto-enable web search
        # just because a key happens to be exported in this environment.
        env_overrides["WEB_SEARCH_ENABLED"] = "0"

print("Resolved environment overrides for run_live.sh:")
for k, v in env_overrides.items():
    print(f"  {k}={'***' if k == 'WEB_SEARCH_API_KEY' else v}")


## Step 14 — Launch the live server

Runs `bash run_live.sh` in the background (it sources the venv itself), logs
to `live_server.log`, and records the PID for the stop/recovery cells below.


In [ ]:
launch_env = os.environ.copy()
launch_env.update(env_overrides)

log_file = open(LOG_PATH, "w")
live_proc = subprocess.Popen(
    ["bash", "run_live.sh"], cwd=PROJECT_ROOT, env=launch_env,
    stdout=log_file, stderr=subprocess.STDOUT,
)
with open(PID_PATH, "w") as f:
    f.write(str(live_proc.pid))

print(f"[ok] launched live server: pid={live_proc.pid}")
print(f"     log file: {LOG_PATH}")
print(f"     pid file: {PID_PATH}")


## Step 15 — Wait for healthy startup

Polls the log for the documented healthy markers and the listening port
simultaneously. On timeout or early process exit, it dumps the log tail and
common-failure hints (CUDA OOM, missing path, traceback) instead of hanging
forever.


In [ ]:
HEALTHY_MARKERS = {
    "lora loaded": "lora loaded",
    "using direct Moshi reply hidden": "using direct Moshi reply hidden",
    "installed PersonaPlex graphed hidden capture": "installed PersonaPlex graphed hidden capture",
    "Uvicorn running on host:port": f"Uvicorn running on http://{HOST}:{PORT}",
    "serving static html": "index_v3_binary_fullscreen_aj_nodrop.html",
}

ERROR_HINTS = {
    "CUDA out of memory": "GPU out of memory — reduce concurrent load or confirm the pod truly has an RTX 5090 with enough VRAM",
    "Traceback (most recent call last)": "a Python exception occurred during startup — see the traceback above in the log",
    "No such file or directory": "a referenced checkpoint/asset path is wrong — re-run Step 10's asset table",
    "CUDA error": "a CUDA/driver mismatch occurred — re-check Step 2 (nvidia-smi) and the torch CUDA build",
    "Address already in use": "the port was taken by another process after the Step 12 check — re-run Step 12",
    "Cannot find": "the VOICE_PROMPT file was not found under VOICE_PROMPT_DIR — re-run Step 10 to fetch/extract voices.tgz",
    "Missing required path": "start_winner_live.sh could not find a required checkpoint/prompt/HTML path — re-run Step 10 and check its printed list",
}


def wait_for_healthy(timeout=STARTUP_TIMEOUT_SEC, poll_interval=POLL_INTERVAL_SEC):
    start = time.time()
    last_print = 0.0
    while time.time() - start < timeout:
        if live_proc.poll() is not None:
            tail_log(120)
            raise RuntimeError(
                f"live server process exited early with code {live_proc.returncode}; see log tail above"
            )

        log_text = ""
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, "r", errors="ignore") as f:
                log_text = f.read()

        markers_ok = {label: (needle in log_text) for label, needle in HEALTHY_MARKERS.items()}
        port_ok = port_listening("127.0.0.1", PORT)

        if all(markers_ok.values()) and port_ok:
            print("[ok] live server is healthy")
            for label, ok in markers_ok.items():
                print(f"  [OK] {label}")
            print(f"  [OK] port {PORT} listening")
            return True

        if time.time() - last_print > 15:
            elapsed = int(time.time() - start)
            print(f"[..] waiting ({elapsed}s/{timeout}s) markers={markers_ok} port_listening={port_ok}")
            last_print = time.time()

        time.sleep(poll_interval)

    print("[error] timed out waiting for healthy startup; log tail:")
    tail_log(150)
    log_text = ""
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, "r", errors="ignore") as f:
            log_text = f.read()
    for needle, hint in ERROR_HINTS.items():
        if needle in log_text:
            print(f"[diagnosis] found '{needle}' in log -> {hint}")
    raise TimeoutError("Live server did not become healthy within the timeout; see diagnostics above")


wait_for_healthy()


## Step 16 — HTTP / Uvicorn confirmation

Final confirmation that the FastAPI/Uvicorn app is actually serving HTTP on
`0.0.0.0:{PORT}` (default `8998`), not just that the port is open. This only
checks `127.0.0.1` inside the pod — it does **not** prove the RunPod public
proxy URL works, since that depends on `PORT` being one of the ports actually
exposed by your pod template (see the note in the Parameters cell). This cell
prints the expected public URL from `RUNPOD_POD_ID` + `PORT` for convenience.


In [ ]:
import urllib.error
import urllib.request

try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=10) as resp:
        body = resp.read(2000).decode(errors="ignore")
        print(f"[ok] HTTP {resp.status} from http://127.0.0.1:{PORT}/")
        print(body[:500])
except urllib.error.URLError as e:
    tail_log(80)
    raise RuntimeError(f"HTTP check failed against http://127.0.0.1:{PORT}/: {e}")

with open(LOG_PATH, "r", errors="ignore") as f:
    final_log = f.read()
assert f"Uvicorn running on http://{HOST}:{PORT}" in final_log, "Uvicorn startup banner not found in log"

print()
print("=" * 72)
print("SUCCESS: PersonaPlex + IMTalker live server (AH AudioPace) is running and healthy")
print(f"  Internal: http://{HOST}:{PORT}")
if PUBLIC_URL:
    print(f"  Public:   {PUBLIC_URL}")
    print(f"  (auto-built from RUNPOD_POD_ID={RUNPOD_POD_ID} and PORT={PORT} — confirm this exact")
    print(f"   port is listed under this pod's Connect -> HTTP Service on the RunPod dashboard;")
    print(f"   the proxy only forwards ports that were exposed when the pod was created/edited)")
else:
    print(f"  Open port {PORT} through your RunPod pod's proxy/port mapping to access the browser UI")
    print(f"  (RUNPOD_POD_ID was not auto-detected; set it in the Parameters cell for a printed URL)")
print(f"  Browser UI file: static/index_v3_binary_fullscreen_aj_nodrop.html")
print(f"  PID: {open(PID_PATH).read().strip()}   Log: {LOG_PATH}")
print("=" * 72)


In [ ]:
# Search/router status report - informational only, never gates the health
# check above. Loading failures are logged and degrade gracefully (the avatar
# still boots and still talks, just without routing/search); this surfaces
# whether that happened AND, critically, prints the actual failure line(s)
# with surrounding context wherever they are in the log -- not just the tail,
# which for a long-running server almost certainly no longer contains the
# startup-time failure by the time you look.
if ENABLE_SEARCH:
    with open(LOG_PATH, "r", errors="ignore") as f:
        _log_lines = f.readlines()
    _log_text = "".join(_log_lines)
    _markers = {
        "reference LoRA loaded": "reference LoRA loaded" in _log_text,
        "STT model loaded": "STT model loaded" in _log_text,
        "context compressor ready": "[compressor] ready" in _log_text,
        "query router ready": "query router ready" in _log_text,
    }
    print("Search pipeline status:")
    for label, ok in _markers.items():
        print(f"  [{'OK' if ok else '??'}] {label}")

    if not _markers["query router ready"]:
        print("\n[warn] the query router never came up -- every turn will be answered from the")
        print("       model's own knowledge and nothing will ever be searched. The router shares")
        print("       the compressor's model, so check the compressor lines first.")

    # Tokenizer identity. If these two ever print the same vocab size, the STT
    # text stream is being decoded with PersonaPlex's vocabulary instead of its
    # own -- which produces real-looking transcripts in the wrong language.
    _tok = [l for l in _log_lines if "stt tokenizer=" in l]
    if _tok:
        print("\nTokenizers:")
        for l in _tok[-1:]:
            print("  " + l.strip())
    _tokwarn = [l for l in _log_lines if "tokenizers look identical" in l]
    if _tokwarn:
        print("  [!!] " + _tokwarn[-1].strip())

    # Transcripts thrown away by the script guard.
    _rejected = [l for l in _log_lines if "| REJECT" in l]
    if _rejected:
        print(f"\n[warn] {len(_rejected)} transcript(s) were rejected as unusable "
              f"(wrong script for an en/fr STT model). Most recent:")
        for l in _rejected[-3:]:
            print("  " + l.rstrip())
        print("  -> compare the two tokenizer vocab sizes printed above; if they match, the")
        print("     STT stream is being decoded with the wrong vocabulary.")

    # Find every "disabled" / "failed" / "[error]" line WHEREVER it is in the
    # log, with context -- fixes the failure mode where the real error had
    # scrolled out of a tail(60)/tail(100) window.
    _needles = ("search disabled", "compressor disabled", "router disabled",
                "[conversation.", "kind=error")
    _hit_line_nums = [i for i, line in enumerate(_log_lines) if any(n in line for n in _needles)]
    if _hit_line_nums:
        print(f"\n[warn] found {len(_hit_line_nums)} error/disable line(s) in the log "
              f"(showing each with 5 lines of context):")
        _shown = set()
        for ln in _hit_line_nums:
            start, end = max(0, ln - 2), min(len(_log_lines), ln + 6)
            if start in _shown:
                continue
            _shown.add(start)
            print(f"  --- log lines {start}-{end} ---")
            for l in _log_lines[start:end]:
                print("  " + l.rstrip())
    else:
        print("\n[ok] no search-related disable/error lines found anywhere in the log.")
else:
    print("ENABLE_SEARCH is False -- routing/search not requested for this launch.")


## Conversation log viewer (run any time during/after a conversation)

Prints the structured conversation log written by `IMTalker/conversation_logger.py`
(user transcripts, router search / no-search decisions with their scores, web
search results, compressor prompts/responses, injected `<ref>`/`<lookup>`
blocks, assistant responses, periodic system status, and errors with full
tracebacks) - both a running human-readable tail and a per-event-kind summary
parsed from the `.jsonl` file. Re-run this cell any time while chatting to see
what actually happened, not just what was loaded at startup.

The `detailed_<session>.log` file in the same directory tells the same story in
plain English, one section per turn, including *why* the router decided the way
it did.


In [ ]:
# Per-turn conversation trace. Reads the JSONL written by
# IMTalker/conversation_logger.py and replays each turn in order:
#   HEARD -> DECIDE (search or not, and why) -> SEARCH -> GROUND -> DONE -> REPLIED
# Re-run any time while chatting.
import glob
import json as _json
from collections import Counter, defaultdict

if ENABLE_SEARCH and CONVERSATION_LOG_DIR:
    jsonl_files = sorted(glob.glob(os.path.join(CONVERSATION_LOG_DIR, "conversation_*.jsonl")))
    if not jsonl_files:
        print(f"[info] no conversation_*.jsonl files yet under {CONVERSATION_LOG_DIR} "
              f"(nothing has happened in a conversation yet, or the server hasn't started).")
    else:
        latest = jsonl_files[-1]
        print(f"Reading {latest}\n")
        events = []
        with open(latest, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    events.append(_json.loads(line))
                except _json.JSONDecodeError:
                    continue

        # ---- headline counts -------------------------------------------
        counts = Counter(e.get("kind", "?") for e in events)
        decisions = [e for e in events if e.get("kind") == "router_decision"]
        rejected = [e for e in events if e.get("kind") == "transcript_rejected"]
        n_search = sum(1 for e in decisions if e.get("needs_search"))
        print(f"Total events : {len(events)}")
        print(f"Turns routed : {len(decisions)}  "
              f"({n_search} searched online, {len(decisions) - n_search} answered directly)")
        if decisions:
            by_src = Counter(e.get("source", "?") for e in decisions)
            print(f"Decided by   : " + ", ".join(f"{k}={v}" for k, v in by_src.most_common()))
        if rejected:
            print(f"[warn] {len(rejected)} transcript(s) rejected as unusable script")

        # ---- per-turn replay --------------------------------------------
        turns = defaultdict(list)
        for e in events:
            t = e.get("turn", e.get("turn_epoch"))
            if t is not None:
                turns[t].append(e)

        LABEL = {
            "user_transcript": "HEARD",
            "transcript_rejected": "REJECT",
            "router_decision": "DECIDE",
            "turn_search": "SEARCH",
            "turn_ground": "GROUND",
            "turn_done": "DONE",
            "assistant_reply": "REPLIED",
        }
        print("\n--- last 5 turns ---")
        for t in sorted(turns)[-5:]:
            print(f"\nTURN {t}")
            for e in turns[t]:
                k = e.get("kind")
                lab = LABEL.get(k)
                if not lab:
                    continue
                if k == "user_transcript":
                    body = f'"{e.get("transcript", "")}"'
                elif k == "transcript_rejected":
                    st = e.get("script_stats") or {}
                    body = (f'DISCARDED ({st.get("non_latin_ratio", 0):.0%} non-Latin) '
                            f'"{str(e.get("transcript", ""))[:60]}"  ids={(e.get("token_ids") or [])[:12]}')
                elif k == "router_decision":
                    body = (f'{"SEARCH ONLINE" if e.get("needs_search") else "ANSWER DIRECTLY":<15} '
                            f'via={e.get("source", "?"):<6} score={e.get("score", 0):.3f}\n'
                            f'{"":<12}why: {e.get("reason", "")}')
                elif k == "turn_search":
                    body = (f'{e.get("provider", "?")}: {e.get("n_found", 0)} found, '
                            f'{e.get("n_kept", 0)} kept ({e.get("elapsed_s", 0):.2f}s)')
                elif k == "turn_ground":
                    body = f'"{e.get("grounding", "")}"' + (" (extractive)" if e.get("used_fallback") else "")
                elif k == "turn_done":
                    body = f'{e.get("outcome", "")} ({e.get("total_s", 0):.2f}s)'
                else:
                    body = f'"{str(e.get("response", ""))[:110]}"'
                print(f"  {lab:<8} {body}")

        errors = [e for e in events if e.get("kind") == "error"]
        if errors:
            print(f"\n--- {len(errors)} error event(s) -- most recent traceback ---")
            print(errors[-1].get("traceback", "")[-3000:])
else:
    print("ENABLE_SEARCH is False, or CONVERSATION_LOG_DIR is empty -- nothing to show.")


## Operational cells (run any time)

These are safe to re-run independently after the server is up.


In [ ]:
# Tail the live server log
tail_log(200)


In [ ]:
# Full diagnostics — safe to run any time, including during a failure
def run_full_diagnostics():
    print("== GPU ==")
    run("nvidia-smi", check=False)

    print("\n== Torch / CUDA (venv) ==")
    try:
        print(get_torch_info())
    except Exception as e:
        print(f"[error] torch check failed: {e}")

    print(f"\n== Port {PORT} ==")
    print(f"listening: {port_listening('127.0.0.1', PORT)}  pids: {find_pids_on_port(PORT)}")

    print("\n== Required assets ==")
    for asset in REQUIRED_ASSETS:
        print(f"  [{'OK' if os.path.exists(asset['path']) else 'MISSING'}] {asset['path']}")
    print(f"  [{'OK' if os.path.exists(VOICE_PROMPT_PATH) else 'MISSING'}] {VOICE_PROMPT_PATH}")

    print("\n== pip check (venv) ==")
    run(f"{VENV_ACTIVATE} && pip check", check=False)

    print("\n== Log tail ==")
    tail_log(100)


run_full_diagnostics()


In [ ]:
# Stop switch — set STOP_SERVER = True and re-run this cell to terminate the live server.
STOP_SERVER = False


def stop_server():
    if not os.path.exists(PID_PATH):
        print("[info] no pid file found; nothing to stop")
        return
    pid = open(PID_PATH).read().strip()
    print(f"[stop] terminating live server pid {pid}")
    subprocess.run(["kill", "-9", pid])
    os.remove(PID_PATH)
    print("[ok] server stopped")


if STOP_SERVER:
    stop_server()
else:
    print("STOP_SERVER is False — set it to True above and re-run this cell to stop the server.")
